In [1]:
import pandas as pd

# 파일 경로
path_v2 = r"C:\Users\Kunny\Research\Project\BiConVarNet\BARD1\urn_mavedb_00000097-0-2_scores.csv"
path_v1 = r"C:\Users\Kunny\Research\Project\BiConVarNet\BARD1\urn_mavedb_00000097-0-1_scores.csv"

# 데이터 로드
df_v2 = pd.read_csv(path_v2)
df_v1 = pd.read_csv(path_v1)

# accession 앞부분 통일 ( -0-1 vs -0-2 차이 제거 )
df_v1["accession"] = df_v1["accession"].str.replace("-0-1", "-0-2", regex=False)

# merge (inner join)
merged = pd.merge(df_v2, df_v1[["accession","hgvs_splice","hgvs_pro","mavedb_clnsig","mavedb_clnrevstat"]],
                  on="accession", how="left")

# 저장
out_path = "BARD1_SGE_merged.tsv"
merged.to_csv(out_path, sep="\t", index=False)

print(f"✅ Merged file saved: {out_path}, shape={merged.shape}")


✅ Merged file saved: BARD1_SGE_merged.tsv, shape=(3893, 12)


In [6]:
merged

,accession,hgvs_nt,score,score_rep1,score_rep2,score_rna,score_rna_rep1,score_rna_rep2,hgvs_splice,hgvs_pro,mavedb_clnsig,mavedb_clnrevstat
0,urn:mavedb:00000097-0-2#1,NM_007294.3:c.5565A>T,-0.015322,-0.116153,0.085509,-0.403450,-0.160708,-0.482648,NM_007294.3:c.5565A>T,NP_009225.1:p.Ile1855=,NaN,NaN
1,urn:mavedb:00000097-0-2#2,NM_007294.3:c.5565A>G,0.021941,0.174501,-0.130620,-0.289526,0.404663,-1.054867,NM_007294.3:c.5565A>G,NP_009225.1:p.Ile1855Met,NaN,NaN
2,urn:mavedb:00000097-0-2#3,NM_007294.3:c.5565A>C,0.231183,0.151333,0.311032,0.207660,0.410202,0.168734,NM_007294.3:c.5565A>C,NP_009225.1:p.Ile1855=,Likely_benign,reviewed_by_expert_panel
3,urn:mavedb:00000097-0-2#4,NM_007294.3:c.5564T>G,-0.464328,-0.140845,-0.787812,0.343402,1.098202,-0.569539,NM_007294.3:c.5564T>G,NP_009225.1:p.Ile1855Arg,NaN,NaN
4,urn:mavedb:00000097-0-2#5,NM_007294.3:c.5564T>C,-0.291519,-0.477009,-0.106029,0.303770,0.111163,0.567510,NM_007294.3:c.5564T>C,NP_009225.1:p.Ile1855Thr,Uncertain_significance,"criteria_provided,_single_submitter"
...,...,...,...,...,...,...,...,...,...,...,...,...
3888,urn:mavedb:00000097-0-2#3889,NM_007294.3:c.40G>T,-1.431762,-0.862572,-2.000952,-0.078645,-0.058883,-0.056970,NM_007294.3:c.40G>T,NP_009225.1:p.Val14Phe,NaN,NaN
3889,urn:mavedb:00000097-0-2#3890,NM_007294.3:c.40G>C,-0.548240,-0.786330,-0.310150,-0.200813,-0.447517,0.007455,NM_007294.3:c.40G>C,NP_009225.1:p.Val14Leu,NaN,NaN
3890,urn:mavedb:00000097-0-2#3891,NM_007294.3:c.40G>A,-0.006564,-0.143607,0.130478,-0.288900,-0.540019,-0.077999,NM_007294.3:c.40G>A,NP_009225.1:p.Val14Ile,NaN,NaN
3891,urn:mavedb:00000097-0-2#3892,NM_007294.3:c.39T>G,-0.299075,-0.461439,-0.136711,-0.068298,-0.003306,-0.084739,NM_007294.3:c.39T>G,NP_009225.1:p.Asn13Lys,NaN,NaN


In [14]:
import pandas as pd
import re
from collections import Counter

# 입력: merged 파일
in_path = r"C:\Users\Kunny\Research\Project\BiConVarNet\BARD1\BARD1_SGE_merged.tsv"

# 출력 파일
out_path_ss = "BARD1_SGE_SS.tsv"
out_path_ra = "BARD1_SGE_RA.tsv"

UNIPROT_ID = "P38398"
STRUCT_FILE = "AF-P38398-F1-model_v4.pdb"

AA3_TO_AA1 = {
    'Ala':'A','Cys':'C','Asp':'D','Glu':'E','Phe':'F',
    'Gly':'G','His':'H','Ile':'I','Lys':'K','Leu':'L',
    'Met':'M','Asn':'N','Pro':'P','Gln':'Q','Arg':'R',
    'Ser':'S','Thr':'T','Val':'V','Trp':'W','Tyr':'Y',
    'Ter':'*'
}

def parse_hgvs_pro(hgvs):
    if pd.isna(hgvs) or not "p." in hgvs:
        return None, None, None
    hgvs = hgvs.split("p.")[-1]
    match = re.match(r"([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2}|\*|=)", hgvs)
    if not match:
        return None, None, None
    wt3, pos, mut3 = match.groups()
    wt = AA3_TO_AA1.get(wt3, None)
    if mut3 == "=":  # synonymous
        mut = wt
    else:
        mut = AA3_TO_AA1.get(mut3, None)
    return wt, int(pos), mut

# 데이터 불러오기
df = pd.read_csv(in_path, sep="\t")

records_ss, records_ra = [], []
excluded = []  # 파싱 실패 저장

for _, row in df.iterrows():
    wt, pos, mut = parse_hgvs_pro(row["hgvs_pro"])
    if wt is None or mut is None:
        excluded.append(row["hgvs_pro"])
        continue
    
    if wt and mut and not pd.isna(row["score"]):
        records_ss.append({
            "UniProtID": UNIPROT_ID,
            "MutPos": pos,
            "WT": wt,
            "Mut": mut,
            "Label": row["score"],
            "StructureFile": STRUCT_FILE,
            "MutPos(pdb)": pos
        })

    if wt and mut and not pd.isna(row["score_rna"]):
        records_ra.append({
            "UniProtID": UNIPROT_ID,
            "MutPos": pos,
            "WT": wt,
            "Mut": mut,
            "Label": row["score_rna"],
            "StructureFile": STRUCT_FILE,
            "MutPos(pdb)": pos
        })

    if wt is None or mut is None or (pd.isna(row["score"]) and pd.isna(row["score_rna"])):
        excluded.append(row["hgvs_pro"])

# DataFrame 생성
out_df_ss = pd.DataFrame(records_ss).sort_values(by="MutPos").reset_index(drop=True)
out_df_ra = pd.DataFrame(records_ra).sort_values(by="MutPos").reset_index(drop=True)

# 저장
out_df_ss.to_csv(out_path_ss, sep="\t", index=False)
out_df_ra.to_csv(out_path_ra, sep="\t", index=False)

# 로그 출력
print(f"총 변이 개수: {len(df)}")
print(f"성공적으로 파싱된 변이(ss): {len(records_ss)}")
print(f"성공적으로 파싱된 변이(ra): {len(records_ra)}")
print(f"제외된 변이: {len(excluded)}")

# 제외된 변이 타입 요약
if excluded:
    type_counts = Counter([str(e).split(":")[-1] if pd.notna(e) else "NA" for e in excluded])
    print("❌ 제외된 변이 타입 예시:")
    for t, c in type_counts.most_common(10):
        print(f"  {t}: {c}개")


총 변이 개수: 3893
성공적으로 파싱된 변이(ss): 2803
성공적으로 파싱된 변이(ra): 2687
제외된 변이: 1090
❌ 제외된 변이 타입 예시:
  NA: 1090개
